# Project 4: OCR Pipeline with Tesseract and OpenCV
**Goal:** Extract machine-readable text from a raw image and validate it with at least 80% confidence.

**Pipeline:** Load image -> Grayscale -> Gaussian Blur -> Adaptive Threshold -> Deskew -> Tesseract OCR -> 80% confidence filter -> Visual output

## 1. Install dependencies

In [ ]:
!apt-get install -y tesseract-ocr > /dev/null
!pip install -q pytesseract opencv-python

## 2. Imports and settings

In [ ]:
import cv2
import numpy as np
import pytesseract
import matplotlib.pyplot as plt
from google.colab import files

CONF_THRESHOLD = 80   # minimum confidence (%) to keep a word
PSM_MODE = 6          # 3=auto, 6=text block, 7=single line, 11=sparse text (invoices)

## 3. Upload an image
Use a sharp, well-lit photo of printed text.

In [ ]:
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
image = cv2.imread(image_path)

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title("Original image")
plt.axis("off")
plt.show()

## 4. Pre-processing: grayscale, Gaussian blur, adaptive thresholding

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)
thresh = cv2.adaptiveThreshold(
    blur, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    31, 15,
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img, title in zip(axes, [gray, blur, thresh],
                          ["1. Grayscale", "2. Gaussian Blur", "3. Adaptive Threshold"]):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.show()

## 5. Deskew (straighten tilted text)

In [ ]:
def deskew(binary):
    coords = np.column_stack(np.where(binary < 128))
    if len(coords) < 50:
        return binary
    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    if abs(angle) < 0.5:
        return binary
    h, w = binary.shape
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(binary, M, (w, h), flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_REPLICATE)

processed = deskew(thresh)
plt.figure(figsize=(8, 8))
plt.imshow(processed, cmap="gray")
plt.title("Final pre-processed image")
plt.axis("off")
plt.show()

## 6. Run OCR and apply the 80% confidence gate

In [ ]:
data = pytesseract.image_to_data(
    processed, config=f"--psm {PSM_MODE}", output_type=pytesseract.Output.DICT
)

results = []
for i, text in enumerate(data["text"]):
    text = text.strip()
    conf = float(data["conf"][i])
    if text and conf >= CONF_THRESHOLD:       # keep only confident words
        results.append({
            "text": text,
            "conf": conf,
            "box": (data["left"][i], data["top"][i], data["width"][i], data["height"][i]),
        })

print("=== OCR RESULT (confidence >= 80%) ===")
print(" ".join(r["text"] for r in results))

if results:
    avg = sum(r["conf"] for r in results) / len(results)
    print(f"\nWords kept: {len(results)}")
    print(f"Average confidence: {avg:.1f}%")
    print("PASS: validated confidence >= 80%" if avg >= CONF_THRESHOLD
          else "FAIL: try a clearer image or another PSM mode")
else:
    print("No words passed the 80% threshold. Try a sharper image or change PSM_MODE.")

## 7. Visual output: boxes and labels

In [ ]:
# Boxes are drawn on the original image, so this assumes no large deskew rotation.
out = image.copy()
for r in results:
    x, y, w, h = r["box"]
    cv2.rectangle(out, (x, y), (x + w, y + h), (0, 200, 0), 2)
    cv2.putText(out, f'{r["text"]} {r["conf"]:.0f}%', (x, max(y - 5, 12)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
plt.title("Final output: detected text with confidence")
plt.axis("off")
plt.show()

cv2.imwrite("final_output.png", out)
files.download("final_output.png")

## Conclusion
This pipeline converts a raw image into clean text using grayscale conversion, Gaussian blur, adaptive thresholding and Tesseract OCR, and keeps only results with at least 80% confidence.